# Notebook 0 — Análise Exploratória: Universo de Empresas Selecionadas

**Objetivo:** Entender quem são as ~214 empresas elegíveis para análise financeira — distribuição setorial, geográfica e estrutura de controle — antes de processar qualquer demonstrativo.  
**Input:** `layer_02_silver.n0_empresas_selecionadas`  
**Output:** Visualizações exploratórias (sem escrita no banco)

## 🧮 Por que excluímos empresas do setor financeiro?

A exclusão de bancos, seguradoras e financeiras **não é uma preferência arbitrária** — é uma exigência técnica de **comparabilidade contábil**. Cada tipo de empresa segue padrões contábeis completamente distintos:

| Tipo de Empresa | Padrão Contábil | Por que não é comparável? |
|:---|:---|:---|
| Indústrias, Varejo, Serviços | **IFRS / BR GAAP** | Padrão base da nossa análise |
| **Bancos** | **COSIF** (Banco Central) | Sem Ativo Circulante tradicional; receita = *Margem Financeira*, não *Receita Bruta* |
| **Seguradoras** | **SUSEP** | Demonstrativos usam *Prêmios Ganhos* e *Sinistros*; sem EBITDA comparável |
| **Financeiras** | Misto COSIF/IFRS | Carteira de crédito como ativo principal — estrutura de BP completamente distinta |
| **Securitizadoras** | IFRS especializado | Ativo baseado em carteiras de recebíveis (CRI, CRA, FIDC) |
| **Holdings (Adm. Part.)** | IFRS (consolidado) | Resultado dominado por equivalência patrimonial, não por operações reais |

### Impacto prático na análise de valuation:

Calcular múltiplos como **EV/EBITDA** ou **P/L** misturando empresas industriais com bancos produz conclusões fundamentalmente incorretas:
- Um banco não tem *Receita Bruta* no sentido industrial — sua receita é a **Margem Financeira**.
- Uma seguradora não tem *Custos Operacionais* comparáveis — tem **Sinistros** e **Provisões Técnicas**.
- Uma holding de participações tem seu resultado dominado por **equivalência patrimonial**, não por operações reais.

> **Nossa escolha metodológica:** Focamos exclusivamente em empresas **não-financeiras** e **operacionais** listadas na **B3**, garantindo que todos os demonstrativos financeiros sigam a mesma estrutura e possam ser comparados de forma significativa.

---
## 1. Setup: Importações e Conexão com o Banco de Dados

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv

load_dotenv()
print("✅ Bibliotecas importadas com sucesso!")

In [ ]:
def create_db_engine():
    user = quote_plus(os.getenv('DB_USER', 'postgres'))
    password = quote_plus(os.getenv('DB_PASS', 'password'))
    host = os.getenv('DB_HOST', 'localhost')
    port = os.getenv('DB_PORT', '5432')
    dbname = os.getenv('DB_NAME', 'data_lake')
    connection_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(connection_str)

engine = create_db_engine()
print("🔌 Conexão com o banco de dados estabelecida!")

---
## 2. Carregando o Universo Completo da CVM

Antes de aplicar qualquer filtro, carregamos o cadastro completo de companhias da CVM para entender o universo de partida.

In [ ]:
print("📥 Carregando o universo completo de empresas da CVM...")

with engine.connect() as conn:
    df_all = pd.read_sql(
        text("SELECT * FROM layer_01_bronze.cad_cia_aberta"),
        con=conn
    )

print(f"✅ Total de registros no cadastro CVM: {len(df_all):,}")
df_all[["CNPJ_CIA", "DENOM_SOCIAL", "SIT", "TP_MERC", "SIT_EMISSOR", "SETOR_ATIV", "UF", "CONTROLE_ACIONARIO"]].head()

---
## 3. O Funil de Filtragem: Como Chegamos ao Nosso Universo de Análise

Cada filtro da query é aplicado de forma incremental, permitindo visualizar exatamente quantas empresas são eliminadas em cada etapa — e o **motivo** de cada exclusão.

In [ ]:
print("🔍 Construindo o funil de filtragem passo a passo...")

passos = []

# Passo 0: Universo completo
passos.append({"filtro": "0. Universo CVM Completo", "n": len(df_all), "excluidas": 0,
               "motivo": "Todos os registros no cadastro da CVM"})

# Passo 1: SIT = 'ATIVO'
df1 = df_all[df_all["SIT"] == "ATIVO"]
passos.append({"filtro": "1. Situação: ATIVO", "n": len(df1), "excluidas": len(df_all) - len(df1),
               "motivo": "Remove canceladas, extintas e suspensas"})

# Passo 2: TP_MERC = 'BOLSA'
df2 = df1[df1["TP_MERC"] == "BOLSA"]
passos.append({"filtro": "2. Mercado: BOLSA", "n": len(df2), "excluidas": len(df1) - len(df2),
               "motivo": "Apenas companhias listadas na B3"})

# Passo 3: SIT_EMISSOR = 'FASE OPERACIONAL'
df3 = df2[df2["SIT_EMISSOR"] == "FASE OPERACIONAL"]
passos.append({"filtro": "3. Emissor: FASE OPERACIONAL", "n": len(df3), "excluidas": len(df2) - len(df3),
               "motivo": "Remove pré-operacionais e em registro"})

# Passo 4: Excluir Holdings
df4 = df3[~df3["SETOR_ATIV"].str.contains("Emp. Adm. Part", na=False, regex=False)]
passos.append({"filtro": "4. Excl. Holdings (Adm. Part.)", "n": len(df4), "excluidas": len(df3) - len(df4),
               "motivo": "Resultado dominado por equivalência patrimonial"})

# Passo 5: Excluir Bancos
df5 = df4[~df4["SETOR_ATIV"].str.contains("Banc", na=False, regex=False)]
passos.append({"filtro": "5. Excl. Bancos", "n": len(df5), "excluidas": len(df4) - len(df5),
               "motivo": "Padrão COSIF — incompatível com análise industrial"})

# Passo 6: Excluir Seguradoras
df6 = df5[~df5["SETOR_ATIV"].str.contains("Segurad", na=False, regex=False)]
passos.append({"filtro": "6. Excl. Seguradoras", "n": len(df6), "excluidas": len(df5) - len(df6),
               "motivo": "Padrão SUSEP — sem EBITDA comparável"})

# Passo 7: Excluir Financeiras
df7 = df6[~df6["SETOR_ATIV"].str.contains("Financeira", na=False, regex=False)]
passos.append({"filtro": "7. Excl. Financeiras", "n": len(df7), "excluidas": len(df6) - len(df7),
               "motivo": "Carteira de crédito como ativo principal"})

# Passo 8: Excluir Securitizadoras
df8 = df7[~df7["SETOR_ATIV"].str.contains("Securitiz", na=False, regex=False)]
passos.append({"filtro": "8. Excl. Securitizadoras", "n": len(df8), "excluidas": len(df7) - len(df8),
               "motivo": "Estrutura de ativo baseada em recebíveis (CRI/CRA/FIDC)"})

df_funil = pd.DataFrame(passos)
print(f"\n🎯 Universo final: {len(df8):,} empresas comparáveis")
print(f"📉 Total excluído: {len(df_all) - len(df8):,} empresas ({(1 - len(df8)/len(df_all))*100:.1f}% do universo inicial)")
print()
df_funil[["filtro", "n", "excluidas", "motivo"]]

In [ ]:
# Gráfico de Funil interativo
cores = ["#2d0057", "#46008a", "#5e00bc", "#7519cc",
         "#8b35d4", "#a055dc", "#b575e4", "#ca95eb", "#dfb8f2"]

fig = go.Figure(go.Funnel(
    y=df_funil["filtro"].tolist(),
    x=df_funil["n"].tolist(),
    textposition="inside",
    textinfo="value+percent initial",
    opacity=0.9,
    marker={
        "color": cores[:len(df_funil)],
        "line": {"width": 1, "color": "white"}
    },
    connector={"line": {"color": "#ccc", "width": 1, "dash": "dot"}}
))

fig.update_layout(
    title={
        "text": "🔍 Funil de Seleção: Do Universo CVM ao Universo de Análise Financeira",
        "font": {"size": 17},
        "x": 0.5,
        "xanchor": "center"
    },
    height=640,
    font={"family": "Arial", "size": 13},
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin={"l": 250, "r": 80, "t": 80, "b": 40}
)

fig.show()

In [ ]:
# Gráfico de barras: empresas excluídas por etapa
df_excl = df_funil[df_funil["excluidas"] > 0].copy()

fig = px.bar(
    df_excl,
    x="excluidas",
    y="filtro",
    orientation="h",
    text="excluidas",
    color="excluidas",
    color_continuous_scale="Purples",
    hover_data={"motivo": True},
    title="❌ Empresas Eliminadas em Cada Etapa do Filtro",
    labels={"excluidas": "Empresas Eliminadas", "filtro": "Filtro Aplicado", "motivo": "Motivo"}
)

fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(
    height=420,
    showlegend=False,
    coloraxis_showscale=False,
    font={"family": "Arial", "size": 12},
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis={"showgrid": True, "gridcolor": "#eee"},
    yaxis={"categoryorder": "total ascending"},
    margin={"l": 280, "r": 100, "t": 60, "b": 50}
)

fig.show()

---
## 4. Empresas Excluídas: Um Olhar sobre os Setores Financeiros

Quais são exatamente os setores que ficaram de fora? Vamos detalhar as categorias excluídas pelos filtros de setor — lembrando que a exclusão ocorre **após** os filtros de situação cadastral (ou seja, são empresas ativas, listadas e operacionais que foram descartadas por incompatibilidade contábil).

In [ ]:
# Empresas excluídas pelos filtros de setor (a partir de df3: ativas, bolsa, operacionais)
mascara_financeiro = (
    df3["SETOR_ATIV"].str.contains("Emp. Adm. Part", na=False, regex=False) |
    df3["SETOR_ATIV"].str.contains("Banc", na=False, regex=False) |
    df3["SETOR_ATIV"].str.contains("Segurad", na=False, regex=False) |
    df3["SETOR_ATIV"].str.contains("Financeira", na=False, regex=False) |
    df3["SETOR_ATIV"].str.contains("Securitiz", na=False, regex=False)
)
df_excluidas_setor = df3[mascara_financeiro].copy()

print(f"📊 Total de empresas excluídas por setor: {len(df_excluidas_setor):,}")
print(f"   (de um total de {len(df3):,} empresas ativas, listadas e operacionais)")
print()

df_setores_excl = (
    df_excluidas_setor
    .groupby("SETOR_ATIV")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
df_setores_excl

In [ ]:
fig = px.bar(
    df_setores_excl.sort_values("count", ascending=True),
    x="count",
    y="SETOR_ATIV",
    orientation="h",
    text="count",
    color="count",
    color_continuous_scale="Purples",
    title="🚫 Setores Excluídos da Análise (Empresas Ativas e Listadas na B3)",
    labels={"count": "Nº de Empresas", "SETOR_ATIV": "Setor de Atividade"}
)

fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(
    height=560,
    showlegend=False,
    coloraxis_showscale=False,
    font={"family": "Arial", "size": 11},
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis={"showgrid": True, "gridcolor": "#eee"},
    margin={"l": 310, "r": 80, "t": 60, "b": 50}
)

fig.show()

---
## 5. O Universo Final: Análise das Empresas Selecionadas

Agora que entendemos quem ficou de fora, vamos explorar **quem entrou**. Analisamos a composição do nosso universo final de empresas comparáveis.

In [ ]:
df_selecionadas = df8.copy()

print("=" * 65)
print("  📊 RESUMO DO UNIVERSO DE ANÁLISE FINANCEIRA")
print("=" * 65)
print(f"  Total de empresas selecionadas : {len(df_selecionadas):>6,}")
print(f"  Total de setores distintos     : {df_selecionadas['SETOR_ATIV'].nunique():>6,}")
print(f"  Estados representados (UF)     : {df_selecionadas['UF'].nunique():>6,}")
print(f"  Tipos de controle acionário    : {df_selecionadas['CONTROLE_ACIONARIO'].nunique():>6,}")
print(f"  Categorias de registro CVM     : {df_selecionadas['CATEG_REG'].nunique():>6,}")
print("=" * 65)

### 5.1 Distribuição por Setor de Atividade

O **Treemap** abaixo mostra a hierarquia dos setores, com o tamanho de cada bloco proporcional ao número de empresas. Em seguida, o ranking detalhado dos setores.

In [ ]:
df_por_setor = (
    df_selecionadas
    .groupby("SETOR_ATIV")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

fig = px.treemap(
    df_por_setor,
    path=["SETOR_ATIV"],
    values="count",
    color="count",
    color_continuous_scale="Purples",
    title="🗂️ Distribuição das Empresas Selecionadas por Setor de Atividade",
    custom_data=["count"]
)

fig.update_traces(
    texttemplate="<b>%{label}</b><br>%{customdata[0]} empresas",
    hovertemplate="<b>%{label}</b><br>Empresas: %{customdata[0]}<extra></extra>",
    textfont_size=12
)

fig.update_layout(
    height=580,
    font={"family": "Arial"},
    coloraxis_showscale=False,
    margin={"l": 10, "r": 10, "t": 60, "b": 10}
)

fig.show()

In [ ]:
df_top = df_por_setor.sort_values("count", ascending=True)

fig = px.bar(
    df_top,
    x="count",
    y="SETOR_ATIV",
    orientation="h",
    text="count",
    color="count",
    color_continuous_scale="Purples",
    title="📊 Setores por Número de Empresas Selecionadas",
    labels={"count": "Nº de Empresas", "SETOR_ATIV": "Setor de Atividade"}
)

fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(
    height=660,
    showlegend=False,
    coloraxis_showscale=False,
    font={"family": "Arial", "size": 11},
    plot_bgcolor="white",
    paper_bgcolor="white",
    yaxis={"categoryorder": "total ascending"},
    xaxis={"showgrid": True, "gridcolor": "#eee"},
    margin={"l": 300, "r": 80, "t": 60, "b": 50}
)

fig.show()

### 5.2 Distribuição Geográfica por Estado (UF)

Onde estão sediadas as empresas do nosso universo de análise? É esperada uma forte concentração em São Paulo (SP), principal polo econômico e financeiro do Brasil.

In [ ]:
df_por_uf = (
    df_selecionadas
    .groupby("UF")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=True)
)

fig = px.bar(
    df_por_uf,
    x="count",
    y="UF",
    orientation="h",
    text="count",
    color="count",
    color_continuous_scale="Purples",
    title="🗺️ Distribuição Geográfica: Empresas por Estado (UF)",
    labels={"count": "Nº de Empresas", "UF": "Estado (UF)"}
)

fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(
    height=500,
    showlegend=False,
    coloraxis_showscale=False,
    font={"family": "Arial", "size": 12},
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis={"showgrid": True, "gridcolor": "#eee"},
    margin={"l": 60, "r": 80, "t": 60, "b": 50}
)

fig.show()

### 5.3 Controle Acionário

A maioria das empresas do nosso universo é de controle **privado** ou há representação significativa de empresas **públicas** (estatais)?

In [ ]:
df_controle = (
    df_selecionadas
    .groupby("CONTROLE_ACIONARIO")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=True)
)

total_ctrl = df_controle["count"].sum()
df_controle["pct"] = (df_controle["count"] / total_ctrl * 100).round(1)
df_controle["label"] = df_controle.apply(lambda r: f"{r['count']}  ({r['pct']}%)", axis=1)

max_count = df_controle["count"].max()

fig = px.bar(
    df_controle,
    x="count",
    y="CONTROLE_ACIONARIO",
    orientation="h",
    text="label",
    color="count",
    color_continuous_scale="Purples",
    title="👥 Distribuição por Tipo de Controle Acionário",
    labels={"count": "Nº de Empresas", "CONTROLE_ACIONARIO": "Controle Acionário"}
)

fig.update_traces(textposition="outside")
fig.update_layout(
    height=400,
    showlegend=False,
    coloraxis_showscale=False,
    font={"family": "Arial", "size": 13},
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis={"showgrid": True, "gridcolor": "#eee", "range": [0, max_count * 1.4]},
    yaxis={"categoryorder": "total ascending"},
    title={"x": 0.5, "xanchor": "center"},
    margin={"l": 160, "r": 40, "t": 70, "b": 50}
)

fig.show()

### 5.4 Categoria de Registro na CVM

A CVM classifica as companhias abertas em duas categorias:
- **Categoria A:** Pode emitir qualquer tipo de valor mobiliário, incluindo ações de livre negociação em bolsa.
- **Categoria B:** Restrita à emissão de valores mobiliários sem ações de livre negociação.

Como nosso filtro inclui apenas empresas listadas na **Bolsa**, esperamos predominância da Categoria A.

In [ ]:
df_categ = (
    df_selecionadas
    .groupby("CATEG_REG")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

fig = px.bar(
    df_categ,
    x="CATEG_REG",
    y="count",
    text="count",
    color="CATEG_REG",
    color_discrete_sequence=["#5e00bc", "#ca95eb"],
    title="📋 Distribuição por Categoria de Registro na CVM",
    labels={"count": "Nº de Empresas", "CATEG_REG": "Categoria de Registro"}
)

fig.update_traces(texttemplate="%{text}", textposition="outside")
fig.update_layout(
    height=420,
    showlegend=False,
    font={"family": "Arial", "size": 13},
    plot_bgcolor="white",
    paper_bgcolor="white",
    yaxis={"showgrid": True, "gridcolor": "#eee"},
    title={"x": 0.5, "xanchor": "center"},
    margin={"l": 60, "r": 60, "t": 70, "b": 60}
)

fig.show()

---
## 6. Conclusões

Após o processo de filtragem e análise exploratória, chegamos a um entendimento claro sobre a composição do nosso universo de análise financeira.

In [ ]:
setor_top = df_por_setor.iloc[0]
uf_top = df_por_uf.sort_values("count", ascending=False).iloc[0]
controle_top = df_controle.iloc[-1]  # df_controle ordenado ascending=True para o gráfico; -1 = maior

print("=" * 70)
print("  🎯 CONCLUSÕES DA ANÁLISE EXPLORATÓRIA")
print("=" * 70)
print(f"\n  📌 Universo de Partida:")
print(f"     {len(df_all):,} empresas no cadastro completo da CVM")
print(f"\n  🎯 Universo de Chegada:")
print(f"     {len(df8):,} empresas comparáveis")
print(f"     ({len(df8)/len(df_all)*100:.1f}% do total — {(1 - len(df8)/len(df_all))*100:.1f}% excluídos)")
print(f"\n  📊 Composição do Universo Final:")
print(f"     • Setor mais representado : {setor_top['SETOR_ATIV']}")
print(f"                                 ({setor_top['count']} empresas)")
print(f"     • Estado com mais sedes   : {uf_top['UF']} ({uf_top['count']} empresas)")
print(f"     • Controle predominante   : {controle_top['CONTROLE_ACIONARIO']}")
print(f"                                 ({controle_top['count']} empresas, "
      f"{controle_top['count']/len(df8)*100:.0f}% do total)")
print(f"\n  ✅ Este universo é a base de todas as análises financeiras")
print(f"     do pipeline de dados da CVM (BP, DRE e DFC).")
print("=" * 70)